Gradient Boosting Classifier

Part 1: Predicting if a Storm Would Occur using Gradient Boosting Classifier

Model:
    Objective: 
        min Z1 classification error, when predicting tropical storms
    Constraints:
        CO2 emisions
        Monthly temps (Jan to Dec)
        Year
        Month,
        Day

        Month should between Jan-Dec
            If month Jan, Mar, May, Jul, Aug, Oct, Dec
	            day>= 1 && day<=31
            If month Apr, Jun, Sep, Nov
	            day>= 1 && day<=30
            If month Feb
	            day>=1 && day <=28
            If year %4 && year %100 &year %400
		        day>= 1 && day<=29	

Expected output - binary value 1 for tropical storm occured else 0

In [50]:
#imports
import pandas as pd
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

In [34]:
#Reading from the dataset
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

#testing if the reading from the dataset was successful
print(data.head(5))

   Year  MONTH  DAY   LAT  LONG  WIND_KTS  PRESSURE CAT  Shape_Leng Country  \
0  1880      8   11  23.0 -91.9        70         0  H1    0.806226  Mexico   
1  1880      8   11  23.4 -92.6        80         0  H1    0.761577  Mexico   
2  1880      8   11  23.7 -93.3        80         0  H1    0.583095  Mexico   
3  1880      8   12  24.0 -93.8        90         0  H2    0.670820  Mexico   
4  1880      9    6  23.9 -88.6        40         0  TS    0.360555  Mexico   

   ...   Jun   Jul   Aug   Sep   Oct   Nov   Dec  Storm Intensity  \
0  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         56.43582   
1  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.92616   
2  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         46.64760   
3  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.37380   
4  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         14.42220   

   Storm Intensity Label  Wind Speed Squared  
0                      3                4900  
1               

In [43]:
def valid_date(row):
    month = row ["MONTH"]
    day = row ["DAY"]
    year = row["Year"]

    if month < 1 or month > 12:
        return False
    if month in [1,3,5,7,8,10,12] and not (1<=day <= 31):
        return False
    if month in [4,6,9,11] and not (1<= day <=30):
        return False
    if month == 2:
        leap_year = (year %4 == 0 and year % 100 != 0) or (year %400 == 0)
        possible_day = 29 if leap_year else 28

        if not (1 <= day <= possible_day):
            return False
    return True
    
    #removing in valid dates from the dataset 
    #data = data[data.apply(valid_date, axis=1)]

Since our dataset only has data when tropical storms occured inorder to use the classifier to train the models, we'll have to use "dummy values" in order to train the model for the classifier to learn te difference. 

Generating realistic "dummy values" by  using the dates what storms did not occur and initilizing information about the storms to be 0 based on their data types

In [ ]:
data["storm_occured"] = 1

#possible dates
years = range(data["Year"].min(), data["Year"].max())
months = range(1,12)
days = range (1, 31)

#possible locations, 20.0 away from the actural tropical storm location
lats =  np.arange(data["LAT"].min(), data["LAT"].max(), 20.0)
longs = np.arange(data["LONG"].min(), data["LONG"].max(), 20.0)

new_rows = pd.DataFrame(product(years, months, days, lats, longs), columns= ["Year", "MONTH", "DAY", "LAT", "LONG"])

#only keeping the rows with valid dates from the new_rows
new_rows = new_rows[new_rows.apply(valid_date, axis=1)]

#Dropping the dates where a storm actually occured
storm_info = data[["Year", "MONTH", "DAY", "LAT", "LONG"]].drop_duplicates()

#comparing the 2 datasets to find the date that is in the new_rows and not in the cleaned tropical storm dataset
merge_data = pd.merge(new_rows, storm_info, how = "left", on= ["Year", "MONTH", "DAY", "LAT", "LONG"], indicator= "merge")
no_storms = merge_data[merge_data["merge"]== "left_only"].drop(columns=["merge"])

#Adding in initilizing values to the no_storm rows
no_storms["storm_occured"] = 0
no_storms["WIND_KTS"] = 0
no_storms["CAT"] = "NA"
no_storms["Storm Intensity"] = 0.0
no_storms["Storm Intensity Label"] = 0

#Using the avg golbal temps and CO2 levels for the no_storms
for column in ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]:
    if column in data.columns:
        no_storms[column] = data[column].mean()

#joing the no_storms to the dataset with the storms
join_data = pd.concat([data, no_storms], ignore_index= True)

#Returning all the rows in a random order
shuffle_data = join_data.sample(frac=1).reset_index(drop=True)

#Updating the csv with the no_storms
shuffle_data.to_csv("completed_dataset_for_IS_project_25.csv", index= False)

Training the model

In [53]:
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

features = ["Year", "MONTH", "DAY", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
target = "storm_occured"

X= data[features]
y = data[target]

#Spliting the merged dataset into training (70%), validation (15%), and testing (15%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=5, stratify=y) #for balance split
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=5, stratify=y_temp)

#testing splits
print(f"Total size: {len(X)}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")


Total size: 3077735
Train size: 2154414
Validation size: 461660
Test size: 461661


Training the Model

In [57]:
#dropping rows with missing values
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

X_val = X_val.dropna()
y_val = y_val.loc[X_val.index]

print(y_train.value_counts())

#model = GradientBoostingClassifier(random_state=5)
#model.fit(X_train, y_train)

storm_occured
1    36859
Name: count, dtype: int64
